# Lab 07 — 3-Qubit Phase-Flip Code

Protect $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ against a single
phase-flip ($Z$) error by encoding it as $\alpha|{+}{+}{+}\rangle + \beta|{-}{-}{-}\rangle$.
In the Hadamard basis a $Z$ error looks like a bit flip ($|+\rangle \leftrightarrow |-\rangle$),
so this is the bit-flip code (Lab 06) conjugated by $H$.

| Qubit | Role |
|---|---|
| `q[0..2]` | data qubits (`q[0]` holds the input and the recovered state) |
| `a[0]` | ancilla measuring $X_0 X_1$ |
| `a[1]` | ancilla measuring $X_1 X_2$ |

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters

In [ ]:
ALPHA, BETA = 0.8, 0.6   # input state alpha|0> + beta|1> (normalized automatically)
ERROR_QUBIT = 2          # 0, 1, 2, or None for no error

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

from common.qlab import normalize, format_state, qubit_state, fidelity, save_circuit

if ERROR_QUBIT not in (None, 0, 1, 2):
    raise ValueError("ERROR_QUBIT must be 0, 1, 2 or None.")

alpha, beta = normalize(ALPHA, BETA)
backend = AerSimulator(method="statevector")

# Syndrome (c1 c0) -> qubit whose phase was flipped
SYNDROME_TO_QUBIT = {0b01: 0, 0b11: 1, 0b10: 2}

## Build the circuit

In [ ]:
def phase_flip_circuit(alpha, beta, error_qubit):
    q = QuantumRegister(3, "q")
    a = QuantumRegister(2, "a")
    c = ClassicalRegister(2, "c")
    qc = QuantumCircuit(q, a, c)

    qc.prepare_state([alpha, beta], q[0])
    qc.barrier(label="Input State Preparation")

    # ---------- Encoding: a|0> + b|1>  ->  a|+++> + b|---> ----------
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])
    qc.h(q)
    qc.barrier(label="Phase-Flip Encoding")

    # ---------- Error Injection ----------
    if error_qubit is not None:
        qc.z(q[error_qubit])
    qc.barrier(label="Error Injection")

    # ---------- Syndrome Extraction: X0X1 -> a0 ----------
    qc.h(q[0])
    qc.h(q[1])
    qc.cx(q[0], a[0])
    qc.cx(q[1], a[0])
    qc.h(q[0])
    qc.h(q[1])
    qc.barrier(label="First Syndrome Extraction")

    # ---------- Syndrome Extraction: X1X2 -> a1 ----------
    qc.h(q[1])
    qc.h(q[2])
    qc.cx(q[1], a[1])
    qc.cx(q[2], a[1])
    qc.h(q[1])
    qc.h(q[2])
    qc.barrier(label="Second Syndrome Extraction")

    qc.measure(a[0], c[0])
    qc.measure(a[1], c[1])
    qc.barrier(label="Syndrome Measurement")

    qc.reset(a[0])
    qc.reset(a[1])
    qc.barrier(label="Ancilla Reset - Available for Reuse")

    # ---------- Error Correction with Z gates ----------
    for syndrome, qubit in SYNDROME_TO_QUBIT.items():
        with qc.if_test((c, syndrome)):
            qc.z(q[qubit])
    qc.barrier(label="Error Correction")

    # ---------- Decoding ----------
    qc.h(q)
    qc.cx(q[0], q[2])
    qc.cx(q[0], q[1])
    qc.barrier(label="Decoding")

    qc.reset(q[1])
    qc.reset(q[2])
    qc.barrier(label="Unused Qubits Reset - Available for Reuse")

    return qc


def run_phase_flip(alpha, beta, error_qubit):
    """Simulate the code; return (syndrome, recovered alpha, recovered beta)."""
    qc = phase_flip_circuit(alpha, beta, error_qubit)
    qc.save_statevector(label="final")

    result = backend.run(transpile(qc, backend), shots=1).result()
    syndrome = int(next(iter(result.get_counts())), 2)
    alpha_out, beta_out = qubit_state(result.data(0)["final"], qubit=0)

    return syndrome, alpha_out, beta_out


qc = phase_flip_circuit(alpha, beta, ERROR_QUBIT)
save_circuit(qc, "circuit.png", scale=1.0);

## Simulate, detect and correct

In [ ]:
syndrome, alpha_out, beta_out = run_phase_flip(alpha, beta, ERROR_QUBIT)

print("Injected error:", "none" if ERROR_QUBIT is None else f"Z on q[{ERROR_QUBIT}]")
print("Syndrome (c1 c0):", format(syndrome, "02b"))

print("\nError detection result:")
if syndrome == 0:
    print("No phase-flip error detected.")
else:
    flipped = SYNDROME_TO_QUBIT[syndrome]
    print(f"Phase-flip error detected at q[{flipped}].")
    print(f"Z gate applied to q[{flipped}] for correction.")

print("\nOriginal state :", format_state(alpha, beta))
print("Recovered state:", format_state(alpha_out, beta_out))
print("Fidelity       :", round(fidelity([alpha, beta], [alpha_out, beta_out]), 10))

## Verification — every single-qubit error position

In [ ]:
for error_qubit in (None, 0, 1, 2):
    syndrome, a_out, b_out = run_phase_flip(alpha, beta, error_qubit)
    f = fidelity([alpha, beta], [a_out, b_out])
    expected = 0 if error_qubit is None else next(s for s, q in SYNDROME_TO_QUBIT.items() if q == error_qubit)

    print(f"error on {str(error_qubit):>4} | syndrome {syndrome:02b} | fidelity {f:.6f}")
    assert syndrome == expected, (error_qubit, syndrome)
    assert np.isclose(f, 1.0), (error_qubit, f)

print("\nPhase-flip code corrected every single Z error.")

In [ ]:
qc.draw(output="mpl", fold=-1, scale=1.0)